# Linear Regression Training Pipeline

This notebook implements an end-to-end Linear Regression training pipeline with:
- Data loading and exploration
- Preprocessing and scaling
- Train-test split
- Model training
- Hyperparameter tuning with Optuna
- Model evaluation and validation

## 1. Import Required Libraries

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import joblib

# Modeling
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, RobustScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
from sklearn.impute import SimpleImputer

# Mute warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load and Explore Dataset

In [18]:
DATA_PATH = '../../data/typhoon_impact_with_extreme_weather.csv' 

# Lead Developer's Configuration
INPUT_FEATURES = [
    'max_sustained_wind_kph',
    'typhoon_type',
    'max_24hr_rainfall_mm',
    'total_storm_rainfall_mm',
    'min_pressure_hpa'
]

# Ordinal Mapping (Better for Linear Regression)
TYPHOON_TYPE_MAPPING = {
    'TD': 0,   'TS': 1,   'STS': 2,  'TY': 3,   'STY': 4
}

try:
    df = pd.read_csv(DATA_PATH)
    
    # Normalize Column Names
    df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('/', '_')
    
    # Target Selection
    TARGET_COLUMN = 'cost' 

    print(f"Data Loaded. Shape: {df.shape}")
    print(f"Target Column: {TARGET_COLUMN}")
    
    # Drop rows where target is missing
    df = df.dropna(subset=[TARGET_COLUMN])
    
    # Check for Zero-Inflation
    zeros = (df[TARGET_COLUMN] == 0).sum()
    print(f"Zero-Cost Events: {zeros} ({zeros/len(df):.1%})")

except Exception as e:
    print(f"Error: {e}")

Data Loaded. Shape: (1776, 30)
Target Column: cost
Zero-Cost Events: 1333 (75.1%)


## 3. Data Preprocessing

In [19]:
X = df[INPUT_FEATURES].copy()
y = df[TARGET_COLUMN].copy()

# 1. Apply Ordinal Encoding
X['typhoon_type'] = X['typhoon_type'].astype(str).str.upper().map(TYPHOON_TYPE_MAPPING)
# Fill missing types with median (usually 1 or 2)
if X['typhoon_type'].isnull().any():
    X['typhoon_type'] = X['typhoon_type'].fillna(X['typhoon_type'].median())

# 2. Add Physics Features (Crucial for Linear Models)
X['wind_power'] = X['max_sustained_wind_kph'] ** 3 
X['pressure_drop'] = 1013 - X['min_pressure_hpa']

# 3. Handle Missing Values
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

# 4. Polynomial Features (The "Curve Fitter")
# This creates interactions like "Wind * Rain" automatically
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly_array = poly.fit_transform(X_imputed)
feature_names = poly.get_feature_names_out(X_imputed.columns)

# Convert back to DF
X_poly = pd.DataFrame(X_poly_array, columns=feature_names, index=X.index)

print(f"Preprocessing Complete.")
print(f"Original Features: {X.shape[1]}")
print(f"Expanded Features (Poly): {X_poly.shape[1]}")

Preprocessing Complete.
Original Features: 7
Expanded Features (Poly): 35


## 4. Train-Test Split

In [20]:
# Use the expanded X_poly dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y, test_size=0.2, random_state=42
)

print(f"Data Split Complete.")

Data Split Complete.


## 5. Feature Scaling

In [21]:
scaler = RobustScaler()

X_train_sc_array = scaler.fit_transform(X_train)
X_test_sc_array = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(
    X_train_sc_array, columns=X_train.columns, index=X_train.index 
)
X_test_scaled = pd.DataFrame(
    X_test_sc_array, columns=X_test.columns, index=X_test.index
)

print("Data Scaled.")

Data Scaled.


## 6. Basic Linear Regression Model Training

In [22]:
print("Training Stage 1 (Logistic Regression)...")

y_train_class = (y_train > 0).astype(int)

baseline_clf = LogisticRegression(
    class_weight='balanced', 
    random_state=42,
    max_iter=1000 # Give it enough time to converge
)
baseline_clf.fit(X_train_scaled, y_train_class)

print("Classifier Trained.")

Training Stage 1 (Logistic Regression)...
Classifier Trained.


## 7. Hyperparameter Tuning with Optuna

In [23]:
mask_nonzero = y_train > 0
X_train_nonzero = X_train_scaled[mask_nonzero]
y_train_nonzero = y_train[mask_nonzero]

# CRITICAL FIX: Transform Target to Log Scale
# Makes 1 Million look like "13.8", preventing the Linear Model from breaking
y_train_log = np.log1p(y_train_nonzero) 

def objective(trial):
    # Tune Alpha (Regularization)
    alpha = trial.suggest_float('alpha', 0.01, 100.0, log=True)
    
    # Ridge is Linear Regression + Safety
    model = Ridge(alpha=alpha, random_state=42)
    
    scores = cross_val_score(
        model, 
        X_train_nonzero, 
        y_train_log, 
        cv=3, 
        scoring='neg_mean_squared_error' 
    )
    return scores.mean()

print("Starting Optuna Tuning (Log-Linear)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

best_params = study.best_params
print(f"Best Alpha: {best_params['alpha']}")

[I 2025-12-11 14:01:49,167] A new study created in memory with name: no-name-d00611a4-c464-4a95-9192-c5afcc21de21
[I 2025-12-11 14:01:49,186] Trial 0 finished with value: -3.7149555238179595 and parameters: {'alpha': 3.495928754900246}. Best is trial 0 with value: -3.7149555238179595.


Starting Optuna Tuning (Log-Linear)...


[I 2025-12-11 14:01:49,202] Trial 1 finished with value: -3.7585617469377035 and parameters: {'alpha': 21.583585600332487}. Best is trial 0 with value: -3.7149555238179595.
[I 2025-12-11 14:01:49,219] Trial 2 finished with value: -3.87092011553481 and parameters: {'alpha': 0.15851591795468167}. Best is trial 0 with value: -3.7149555238179595.
[I 2025-12-11 14:01:49,229] Trial 3 finished with value: -3.711927732800486 and parameters: {'alpha': 5.031061152975635}. Best is trial 3 with value: -3.711927732800486.
[I 2025-12-11 14:01:49,240] Trial 4 finished with value: -3.798068654265679 and parameters: {'alpha': 35.8389489987092}. Best is trial 3 with value: -3.711927732800486.
[I 2025-12-11 14:01:49,248] Trial 5 finished with value: -3.945398437635028 and parameters: {'alpha': 0.03588743095525077}. Best is trial 3 with value: -3.711927732800486.
[I 2025-12-11 14:01:49,257] Trial 6 finished with value: -3.9488119706110325 and parameters: {'alpha': 0.03274012912320454}. Best is trial 3 wit

Best Alpha: 5.031061152975635


In [24]:
print("Best Hyperparameters:")
print(best_params)

Best Hyperparameters:
{'alpha': 5.031061152975635}


## 8. Model Evaluation and Validation

In [25]:
# --- STEP 1: Train Regressor (Log Scale) ---
final_reg = Ridge(**best_params, random_state=42)
final_reg.fit(X_train_nonzero, y_train_log)

# --- STEP 2: Find Best Classifier Threshold ---
# Automatically find the threshold that gives Highest Accuracy
probs_train = baseline_clf.predict_proba(X_train_scaled)[:, 1]
y_train_binary = (y_train > 0).astype(int)

best_threshold = 0.5
best_acc = 0

for t in np.arange(0.1, 0.9, 0.05):
    preds = (probs_train >= t).astype(int)
    acc = accuracy_score(y_train_binary, preds)
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

print(f"Optimized Threshold Found: {best_threshold:.2f} (Train Acc: {best_acc:.2%})")

# --- STEP 3: Make Final Predictions ---
# A. Classifier
probs_test = baseline_clf.predict_proba(X_test_scaled)[:, 1]
pred_is_damage = (probs_test >= best_threshold).astype(int)

# B. Regressor (Log -> Real Money)
pred_log = final_reg.predict(X_test_scaled)
pred_amount = np.expm1(pred_log) 
pred_amount = np.maximum(pred_amount, 0) # Safety clamp

# C. Combine
final_predictions = pred_is_damage * pred_amount

# --- STEP 4: Metrics ---
acc = accuracy_score((y_test > 0).astype(int), pred_is_damage)
mae = mean_absolute_error(y_test, final_predictions)
r2 = r2_score(y_test, final_predictions)

print(f"Final Evaluation (Improved Linear Pipeline):")
print(f"Damage Detection Accuracy: {acc:.2%}")
print(f"MAE: {mae:,.2f}")
print(f"R2 Score: {r2:.4f}")

Optimized Threshold Found: 0.70 (Train Acc: 79.23%)
Final Evaluation (Improved Linear Pipeline):
Damage Detection Accuracy: 77.25%
MAE: 242,785.43
R2 Score: 0.2618


## 10. Model Output and Summary

In [28]:
results_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': final_predictions
})

results_df['Error'] = results_df['Actual'] - results_df['Predicted']
results_df['Abs_Error'] = results_df['Error'].abs()

print("--- Top 10 Predictions (vs Actual) ---")
display(results_df.head(10))

zeros_mask = results_df['Actual'] == 0
zeros_correct = (results_df.loc[zeros_mask, 'Predicted'] == 0).sum()
print(f"Zero-Cost Accuracy: {zeros_correct}/{zeros_mask.sum()}")

--- Top 10 Predictions (vs Actual) ---


,Actual,Predicted,Error,Abs_Error
0,1258696.00,0.000000e+00,1.258696e+06,1.258696e+06
1,0.00,0.000000e+00,0.000000e+00,0.000000e+00
2,0.00,0.000000e+00,0.000000e+00,0.000000e+00
3,0.00,0.000000e+00,0.000000e+00,0.000000e+00
4,0.00,0.000000e+00,0.000000e+00,0.000000e+00
5,0.00,0.000000e+00,0.000000e+00,0.000000e+00
6,0.00,0.000000e+00,0.000000e+00,0.000000e+00
7,0.00,0.000000e+00,0.000000e+00,0.000000e+00
8,2842445.52,3.991292e+06,-1.148846e+06,1.148846e+06
9,0.00,0.000000e+00,0.000000e+00,0.000000e+00


Zero-Cost Accuracy: 257/272


In [27]:
results_df.to_csv('predictions_final_linear_improved.csv', index=False)

# joblib.dump(scaler, 'scaler_linear.joblib')
# joblib.dump(poly, 'poly_features.joblib')
# joblib.dump(baseline_clf, 'classifier_logistic.joblib')
# joblib.dump(final_reg, 'regressor_ridge.joblib')

print("All Improved Linear models saved.")

All Improved Linear models saved.
